# Tutorial 01 — Hello World
## NCHC LLM 工作坊 2026 · NVIDIA NeMo Agent Toolkit

---

### 你會學到什麼

- 什麼是 **NeMo Agent Toolkit workflow**
- 怎麼閱讀把 LLM 串到 tool 的 **YAML config**
- 怎麼執行 `nat run` 並解讀輸出
- 怎麼用 CLI 覆寫 config 數值而不必改檔案

### 一張圖總覽

```
你的問題
      │
      ▼
┌──────────────────────────────┐
│      tool_calling_agent      │
│                              │
│  1. LLM 讀取問題              │
│  2. LLM 呼叫 get_time tool   │
│  3. Tool 傳回時間戳           │
│  4. LLM 寫出最終答案          │
└──────────────────────────────┘
      │
      ▼
    答案
```

**模型：** `nvidia/nemotron-3-nano-30b-a3b`，透過 **NVIDIA NIM** 提供服務

> **開始前：** 從工作坊根目錄執行 `uv sync` 與 `uv run jupyter lab`。

---
## 步驟 1 — 確認環境


In [1]:
import os, subprocess, getpass

result = subprocess.run(["nat", "--version"], capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    key = getpass.getpass("NVIDIA_API_KEY 尚未設定。請輸入你的 key（nvapi-...）：")
    os.environ["NVIDIA_API_KEY"] = key.strip()
print("NVIDIA_API_KEY ✓")

# 用系統時區，讓 datetime tool 回傳本地時間
os.environ["NAT_CONFIG_DIR"] = os.path.abspath("nat_config")
print("NAT_CONFIG_DIR ✓")

nat, version 1.6.0
NVIDIA_API_KEY ✓
NAT_CONFIG_DIR ✓


---
## 步驟 2 — 看 Workflow Config

每個 NeMo Agent Toolkit workflow 都用一份 **YAML 檔**定義，有三個區段：

| 區段 | 用途 |
|---|---|
| `functions` | agent 可以呼叫的 tool |
| `llms` | 語言模型設定 |
| `workflow` | agent 類型 + 把哪些 LLM 與 tool 串起來 |

本範例的 config 給 agent 一個內建 tool：**`current_datetime`** — 不需安裝。

---
## 步驟 3 — 執行 Workflow

`nat run` 接受兩個主要參數：
- `--config_file` — YAML config 的路徑
- `--input` — 你的問題

看 verbose 輸出可以即時觀察 **tool 呼叫**與 **model 回應**。

In [2]:
!nat run \
    --config_file tutorial_01_hello_world/configs/config.yml \
    --input "現在幾點？"

2026-05-19 15:16:11 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'tutorial_01_hello_world/configs/config.yml'
/home/ubuntu/NCHC-Agentic-AI-Bootcamp-2026/04_NeMo_Agent_Toolkit/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-05-19 15:16:11 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-05-19 15:16:20 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:173 - 
------------------------------
[AGENT]
Agent input: What t

### 解讀輸出

log 裡會出現三個階段：

1. **Agent 的思考** — 模型決定呼叫 `get_time`
2. **Tool 的回應** — 內建 tool 傳回的時間戳
3. **Workflow Result** — 模型寫出的最終答案

```
Calling tools: ['get_time']
Tool's response: The current time of day is 2026-05-04 14:21:00 +0800

Workflow Result:
['It is 2:21 PM in Taiwan Standard Time. Hello to the NCHC workshop!']
```

> 你可能會看到 "model type unknown" 警告——這沒關係，不影響執行。

---
## 步驟 4 — 更多範例


In [ ]:
# 這個問題 LLM 無法使用給予的工具回答
!nat run \
    --config_file tutorial_01_hello_world/configs/config.yml \
    --input "甚麼是NVIDIA NIM？"

2026-05-19 15:24:26 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'tutorial_01_hello_world/configs/config.yml'
/home/ubuntu/NCHC-Agentic-AI-Bootcamp-2026/04_NeMo_Agent_Toolkit/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey

Configuration Summary:
--------------------
Workflow Type: tool_calling_agent
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-05-19 15:24:26 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-05-19 15:24:43 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:173 - 
------------------------------
[AGENT]
Agent input: What i

---
## 步驟 5 — 用 CLI 覆寫 Config 數值

不必改 YAML 檔，用 `--override` 就能改任何 config 參數：

```bash
nat run --config_file ... --override <dot.path> <value> --input "..."
```

路徑用點記法：
- `llms.nemotron.temperature` → `llms.nemotron` 內的 `temperature` 欄位
- `workflow.verbose` → `workflow` 內的 `verbose` 欄位

In [ ]:
# 調整 temperature 可以改變回應的多樣性
!nat run \
    --config_file tutorial_01_hello_world/configs/config.yml \
    --override llms.nemotron.temperature 0.9 \
    --input "現在幾點？"

2026-05-19 15:24:45 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'tutorial_01_hello_world/configs/config.yml'
2026-05-19 15:24:45 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for llms.nemotron.temperature with value: 0.9 with type <class 'float'>)
2026-05-19 15:24:45 - INFO     - nat.cli.cli_utils.config_override:211 - 

Configuration after overrides:

functions:
  get_time:
    _type: current_datetime
llms:
  nemotron:
    _type: nim
    max_tokens: 4096
    model_name: nvidia/nemotron-3-nano-30b-a3b
    temperature: 0.9
workflow:
  _type: tool_calling_agent
  llm_name: nemotron
  tool_names:
  - get_time
  verbose: true

/home/ubuntu/NCHC-Agentic-AI-Bootcamp-2026/04_NeMo_Agent_Toolkit/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey

Configur

In [ ]:
# 隱藏中間步驟 — 只顯示最終答案
!nat run \
    --config_file tutorial_01_hello_world/configs/config.yml \
    --override workflow.verbose false \
    --input "現在幾點？"

2026-05-19 15:25:06 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'tutorial_01_hello_world/configs/config.yml'
2026-05-19 15:25:06 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for workflow.verbose with value: False with type <class 'bool'>)
2026-05-19 15:25:06 - INFO     - nat.cli.cli_utils.config_override:211 - 

Configuration after overrides:

functions:
  get_time:
    _type: current_datetime
llms:
  nemotron:
    _type: nim
    max_tokens: 4096
    model_name: nvidia/nemotron-3-nano-30b-a3b
    temperature: 1.0
workflow:
  _type: tool_calling_agent
  llm_name: nemotron
  tool_names:
  - get_time
  verbose: false

/home/ubuntu/NCHC-Agentic-AI-Bootcamp-2026/04_NeMo_Agent_Toolkit/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey

Configuration S

---
## 總結

| 概念 | 學到了什麼 |
|---|---|
| **Workflow config** | 三個區段：`functions`、`llms`、`workflow` |
| **`_type: nim`** | 把 LLM 接到 NVIDIA NIM |
| **`tool_calling_agent`** | 使用 LLM 原生 tool-calling API 的 agent |
| **`nat run`** | 帶問題執行 workflow 的 CLI 指令 |
| **`--override key value`** | 不改 YAML 也能覆寫任何欄位 |
| **要不要用 tool** | agent 會依問題決定是否呼叫 tool |

➡️ **下一個：** `tutorial_02_code_agent.ipynb`